## <a name="head_network">6. head网络</a>

在head网络中，我们完成Proposal的分类、包围框回归以及Mask回归3个分支，其中分类与包围框回归的结构与RPN中Anchor的分类与回归相似，我们先来看一下其结构。


### 分类与包围框回归分支

经过RoI Align层之后，经过两次 $Conv \to BatchNorm \to ReLU$运算，得到一个卷积层结果，然后将其输出到两个全连接层中，得到分类分支和包围框回归分支，因为包围框回归的修正信息为一个四元组（与RPN一样），因此分类分支全连接层的大小即分类的数量，而包围框回归分支全连层的大小为分类数量x4。下面，我们构造head网络中的这两个分支，并显示网络结构：

In [ ]:
import keras.layers as KL
import keras.backend as K


fc_layers_size = 1024
pool_size = 7
train_bn = False
num_classes = 80 + 1

# align layer shape: [batch, num_boxes, pool_height, pool_width, channels]
roi_align = KL.Input(shape=[321, pool_size, pool_size, 512], name='mock_roi_align_layer')
# Two 1024 FC layers (implemented with Conv2D for consistency)
x = KL.TimeDistributed(KL.Conv2D(fc_layers_size, (pool_size, pool_size), padding="valid"),
                       name="mrcnn_class_conv1")(roi_align)
x = KL.TimeDistributed(KL.BatchNormalization(), name='mrcnn_class_bn1')(x, training=train_bn)
x = KL.Activation('relu')(x)
x = KL.TimeDistributed(KL.Conv2D(fc_layers_size, (1, 1)),
                       name="mrcnn_class_conv2")(x)
x = KL.TimeDistributed(KL.BatchNormalization(), name='mrcnn_class_bn2')(x, training=train_bn)
x = KL.Activation('relu')(x)

shared = KL.Lambda(lambda x: K.squeeze(K.squeeze(x, 3), 2),
                   name="pool_squeeze")(x)

# Classifier head
mrcnn_class_logits = KL.TimeDistributed(KL.Dense(num_classes),
                                        name='mrcnn_class_logits')(shared)
mrcnn_probs = KL.TimeDistributed(KL.Activation("softmax"),
                                 name="mrcnn_class")(mrcnn_class_logits)

# BBox head
# [batch, num_rois, NUM_CLASSES * (dy, dx, log(dh), log(dw))]
x = KL.TimeDistributed(KL.Dense(num_classes * 4, activation='linear'),
                       name='mrcnn_bbox_fc')(shared)
# Reshape to [batch, num_rois, NUM_CLASSES, (dy, dx, log(dh), log(dw))]
s = K.int_shape(x)
mrcnn_bbox = KL.Reshape((s[1], num_classes, 4), name="mrcnn_bbox")(x)

print(mrcnn_class_logits)
print(mrcnn_probs)
print(mrcnn_bbox)
print(roi_align.shape)

from keras.models import Model
from keras.utils import plot_model

m = Model([roi_align], [mrcnn_class_logits, mrcnn_probs, mrcnn_bbox])
plot_model(m, to_file='head_classifier_and_bbox.png')

![head_classifier_and_bbox](head_classifier_and_bbox.png)

### Mask分支

RoI Align层将Mask遮罩的尺寸固定池化为14x14，输入到head网络后，先对其进行若干次$conv \to BatchNorm \to ReLU$运算得到卷积结构，然后对结果进行2x2的反卷积操作，得到尺寸为28x28的Mask数据，最后，通过以sigmoid为激活函数的1x1的卷积，将输出的shape深度转为分类个数，同时每个像素的值输出为0或1，代表在28x28的区域内，每个像素是否为遮罩。

In [ ]:
MASK_POOL_SIZE = 14

# Conv layers
roi_align = KL.Input(shape=[321, MASK_POOL_SIZE, MASK_POOL_SIZE, 512], name='mock_roi_align_layer')

x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                       name="mrcnn_mask_conv1")(roi_align)
x = KL.TimeDistributed(KL.BatchNormalization(),
                       name='mrcnn_mask_bn1')(x, training=train_bn)
x = KL.Activation('relu')(x)

x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                       name="mrcnn_mask_conv2")(x)
x = KL.TimeDistributed(KL.BatchNormalization(),
                       name='mrcnn_mask_bn2')(x, training=train_bn)
x = KL.Activation('relu')(x)

x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                       name="mrcnn_mask_conv3")(x)
x = KL.TimeDistributed(KL.BatchNormalization(),
                       name='mrcnn_mask_bn3')(x, training=train_bn)
x = KL.Activation('relu')(x)

x = KL.TimeDistributed(KL.Conv2D(256, (3, 3), padding="same"),
                       name="mrcnn_mask_conv4")(x)
x = KL.TimeDistributed(KL.BatchNormalization(),
                       name='mrcnn_mask_bn4')(x, training=train_bn)
x = KL.Activation('relu')(x)

x = KL.TimeDistributed(KL.Conv2DTranspose(256, (2, 2), strides=2, activation="relu"),
                       name="mrcnn_mask_deconv")(x)

mrcnn_mask = KL.TimeDistributed(KL.Conv2D(num_classes, (1, 1), strides=1, activation="sigmoid"),
                       name="mrcnn_mask")(x)
head_mask_model = Model([roi_align], [mrcnn_mask])
plot_model(head_mask_model, to_file='head_mask_graph.png')

![head_mask_graph](head_mask_graph.png)

In [ ]:
print(mrcnn_mask.shape)

可以看到，mask分支的shape为$[batch, num(masks), 28, 28, 81]$，即81个分类，每个分类都预测一个28x28的Mask信息。在训练时，使用二分类交叉熵作为损失函数，因为Mask分支中，每个元素的值为0或1，代表是否为遮罩。

至此，我们就构建好了head网络的三个分支。